# Yb174 lower-Rydberg lifetimes vs. Fang et al. (2001)

This notebook uses the same notebook-local repair idea as `yb171_total_decay_get_nearby_repair.ipynb`: final states are generated dynamically with `atom.get_nearby(...)`, but the MQDT effective-quantum-number interval is split into small windows before summing `partial_decay`.

The validation target is Fang et al., *Journal of Quantitative Spectroscopy and Radiative Transfer* **69**, 469-473 (2001), DOI: `10.1016/S0022-4073(00)00096-0`. The paper measured neutral ytterbium `6sns ^1S0` and `6snd` Rydberg lifetimes by delayed electric-field ionization and reported room-temperature lifetimes in Table 1.

Important caveat: this is a validation stress test for the current `get_nearby + partial_decay` algorithm. It is not expected to reproduce Fang exactly unless the same final-state basis, low-lying perturbers, blackbody model, and state-mixing conventions are all represented.


In [1]:
from __future__ import annotations

import copy
import math
import os
import sys
import time
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-rydcalc")

from neutral_yb.external.rydcalc_adapter import load_rydcalc  # noqa: E402

rydcalc = load_rydcalc(require_c_extension=True)
yb = rydcalc.Ytterbium174(cpp_numerov=True, use_db=False)
print(yb.name)


174Yb


## Fang et al. Table 1 data used here

The table below encodes the room-temperature values from Fang et al. Table 1 for the rows `6s21s` to `6s27s` and `6s21d` to `6s27d`. Columns:

- `exp_us`: reported experimental lifetime, when present;
- `fang_300K_us`: Fang's calculated effective room-temperature lifetime, `1 / (A + B rho)`;
- `fang_spont_us`: Fang's calculated spontaneous lifetime, `1 / A`.

Some experimental entries are absent in the paper table and are represented by `None`.


In [2]:
FANG_2001 = {
    ("s", 21): {"exp_us": None, "fang_300K_us": 6.8, "fang_spont_us": 7.6},
    ("s", 22): {"exp_us": 6.5, "fang_300K_us": 8.3, "fang_spont_us": 9.1},
    ("s", 23): {"exp_us": 8.1, "fang_300K_us": 9.8, "fang_spont_us": 10.7},
    ("s", 24): {"exp_us": 9.4, "fang_300K_us": 11.2, "fang_spont_us": 12.5},
    ("s", 25): {"exp_us": 10.6, "fang_300K_us": 13.0, "fang_spont_us": 14.5},
    ("s", 26): {"exp_us": 12.1, "fang_300K_us": 14.9, "fang_spont_us": 16.7},
    ("s", 27): {"exp_us": None, "fang_300K_us": 16.9, "fang_spont_us": 19.2},
    ("d", 21): {"exp_us": None, "fang_300K_us": 7.8, "fang_spont_us": 8.6},
    ("d", 22): {"exp_us": 6.4, "fang_300K_us": 9.0, "fang_spont_us": 10.1},
    ("d", 23): {"exp_us": 7.7, "fang_300K_us": 10.6, "fang_spont_us": 11.7},
    ("d", 24): {"exp_us": 8.7, "fang_300K_us": 12.1, "fang_spont_us": 13.5},
    ("d", 25): {"exp_us": 9.8, "fang_300K_us": 13.7, "fang_spont_us": 15.5},
    ("d", 26): {"exp_us": 10.7, "fang_300K_us": 15.9, "fang_spont_us": 17.6},
    ("d", 27): {"exp_us": None, "fang_300K_us": 18.2, "fang_spont_us": 20.1},
}

for series in ("s", "d"):
    print(f"Fang 2001 {series}-series")
    print(" n   exp_us   calc_300K_us   calc_1/A_us")
    for n in range(21, 28):
        row = FANG_2001[(series, n)]
        exp = "--" if row["exp_us"] is None else f"{row['exp_us']:.1f}"
        print(f"{n:2d} {exp:>8} {row['fang_300K_us']:14.1f} {row['fang_spont_us']:13.1f}")
    print()


Fang 2001 s-series
 n   exp_us   calc_300K_us   calc_1/A_us
21       --            6.8           7.6
22      6.5            8.3           9.1
23      8.1            9.8          10.7
24      9.4           11.2          12.5
25     10.6           13.0          14.5
26     12.1           14.9          16.7
27       --           16.9          19.2

Fang 2001 d-series
 n   exp_us   calc_300K_us   calc_1/A_us
21       --            7.8           8.6
22      6.4            9.0          10.1
23      7.7           10.6          11.7
24      8.7           12.1          13.5
25      9.8           13.7          15.5
26     10.7           15.9          17.6
27       --           18.2          20.1



In [3]:
def state_signature(state, *, energy_bin_hz: float = 1e6):
    return (
        round(state.get_energy_Hz() / energy_bin_hz),
        round(2 * state.f),
        round(2 * state.m),
        int(state.parity),
    )


def build_windowed_get_nearby_basis(
    atom,
    initial,
    *,
    nu_min: float = 3.0,
    nu_max: float | None = None,
    window_width: float = 2.0,
    window_overlap: float = 0.05,
    dl: int = 1,
    df: int = 1,
    dm: int = 1,
    dipole_allowed: bool = True,
):
    if nu_max is None:
        nu_max = initial.nub + 10.0

    unique = {}
    errors = []
    skipped = {"nonfinite_energy": 0, "not_dipole_allowed": 0, "duplicates": 0}
    raw_states = 0
    start = time.perf_counter()
    lo = float(nu_min)
    while lo < nu_max - 1e-12:
        hi = min(float(nu_max), lo + float(window_width))
        query_lo = lo - 0.5 * window_overlap
        query_hi = hi + 0.5 * window_overlap
        center = 0.5 * (query_lo + query_hi)
        half_width = 0.5 * (query_hi - query_lo)

        proxy = copy.copy(initial)
        proxy.nub = center
        proxy.v_exact = center
        proxy.n = center
        proxy.v = center

        try:
            states = atom.get_nearby(
                proxy,
                include_opts={"dn": half_width, "dl": dl, "df": df, "dm": dm},
            )
        except Exception as exc:
            states = []
            errors.append(
                {
                    "nu_lo": query_lo,
                    "nu_hi": query_hi,
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                }
            )
        raw_states += len(states)

        for final_state in states:
            if final_state is None:
                continue
            try:
                energy_hz = final_state.get_energy_Hz()
            except Exception:
                skipped["nonfinite_energy"] += 1
                continue
            if not math.isfinite(float(energy_hz)):
                skipped["nonfinite_energy"] += 1
                continue
            if dipole_allowed and not initial.allowed_multipole(final_state, k=1):
                skipped["not_dipole_allowed"] += 1
                continue
            key = state_signature(final_state)
            if key in unique:
                skipped["duplicates"] += 1
                continue
            unique[key] = final_state
        lo = hi

    return {
        "states": list(unique.values()),
        "raw_states": raw_states,
        "errors": errors,
        "skipped": skipped,
        "seconds": time.perf_counter() - start,
    }


def decay_from_basis(atom, initial, basis, *, temperature_k: float):
    env = rydcalc.environment(T_K=temperature_k)
    total_gamma = 0.0
    nonzero = 0
    failures = 0
    start = time.perf_counter()
    for final_state in basis["states"]:
        try:
            gamma = atom.partial_decay(initial, final_state, env)
        except Exception:
            failures += 1
            continue
        if math.isfinite(float(gamma)) and gamma != 0:
            total_gamma += float(gamma)
            nonzero += 1
    return {
        "gamma_s^-1": total_gamma,
        "tau_us": math.inf if total_gamma == 0 else 1e6 / total_gamma,
        "nonzero": nonzero,
        "failures": failures,
        "seconds": time.perf_counter() - start,
    }


In [4]:
def initial_state_for_fang_row(series: str, n: int):
    if series == "s":
        return yb.get_state((n, 0, 0, 0), tt="vlfm")  # 6sns ^1S0, m=0
    if series == "d":
        return yb.get_state((n, 2, 2, 0), tt="vlfm")  # 6snd ^1D2, m=0
    raise ValueError(series)


computed = []
for series in ("s", "d"):
    for n in range(21, 28):
        initial = initial_state_for_fang_row(series, n)
        basis = build_windowed_get_nearby_basis(
            yb,
            initial,
            nu_min=3.0,
            nu_max=initial.nub + 10.0,
            window_width=2.0,
            window_overlap=0.05,
            dl=1,
            df=1,
            dm=1,
            dipole_allowed=True,
        )
        result_0 = decay_from_basis(yb, initial, basis, temperature_k=0.0)
        result_300 = decay_from_basis(yb, initial, basis, temperature_k=300.0)
        fang = FANG_2001[(series, n)]
        computed.append(
            {
                "series": series,
                "n": n,
                "initial": str(initial),
                "nub": float(initial.nub),
                "basis_raw": basis["raw_states"],
                "basis_unique": len(basis["states"]),
                "basis_errors": len(basis["errors"]),
                "tau_0K_us": result_0["tau_us"],
                "tau_300K_us": result_300["tau_us"],
                "fang_exp_us": fang["exp_us"],
                "fang_300K_us": fang["fang_300K_us"],
                "fang_spont_us": fang["fang_spont_us"],
                "ratio_to_fang_300K": result_300["tau_us"] / fang["fang_300K_us"],
            }
        )
        print(
            f"{series}{n}: tau_300K={result_300['tau_us']:.3f} us, "
            f"Fang calc={fang['fang_300K_us']:.1f} us, "
            f"ratio={result_300['tau_us'] / fang['fang_300K_us']:.2f}, "
            f"basis={len(basis['states'])}, errors={len(basis['errors'])}"
        )


s21: tau_300K=12.215 us, Fang calc=6.8 us, ratio=1.80, basis=174, errors=0
s22: tau_300K=13.658 us, Fang calc=8.3 us, ratio=1.65, basis=180, errors=0
s23: tau_300K=15.193 us, Fang calc=9.8 us, ratio=1.55, basis=186, errors=0
s24: tau_300K=16.821 us, Fang calc=11.2 us, ratio=1.50, basis=192, errors=0
s25: tau_300K=18.541 us, Fang calc=13.0 us, ratio=1.43, basis=198, errors=0
s26: tau_300K=20.354 us, Fang calc=14.9 us, ratio=1.37, basis=204, errors=0
s27: tau_300K=22.260 us, Fang calc=16.9 us, ratio=1.32, basis=210, errors=0
d21: tau_300K=15.446 us, Fang calc=7.8 us, ratio=1.98, basis=540, errors=0
d22: tau_300K=17.130 us, Fang calc=9.0 us, ratio=1.90, basis=558, errors=0
d23: tau_300K=18.144 us, Fang calc=10.6 us, ratio=1.71, basis=576, errors=0
d24: tau_300K=21.270 us, Fang calc=12.1 us, ratio=1.76, basis=594, errors=0
d25: tau_300K=23.264 us, Fang calc=13.7 us, ratio=1.70, basis=612, errors=0
d26: tau_300K=25.431 us, Fang calc=15.9 us, ratio=1.60, basis=630, errors=0
d27: tau_300K=27.

In [5]:
def fmt_optional(value):
    return "--" if value is None else f"{value:.1f}"

for series in ("s", "d"):
    print(f"Comparison for {series}-series")
    print(" n  this_300K  Fang_300K  ratio  this_0K  Fang_1/A  exp  unique")
    ratios = []
    for row in [row for row in computed if row["series"] == series]:
        ratios.append(row["ratio_to_fang_300K"])
        print(
            f"{row['n']:2d} "
            f"{row['tau_300K_us']:10.3f} "
            f"{row['fang_300K_us']:10.1f} "
            f"{row['ratio_to_fang_300K']:6.2f} "
            f"{row['tau_0K_us']:8.3f} "
            f"{row['fang_spont_us']:8.1f} "
            f"{fmt_optional(row['fang_exp_us']):>5} "
            f"{row['basis_unique']:6d}"
        )
    print(f"mean ratio to Fang 300 K = {sum(ratios) / len(ratios):.2f}")
    print()


Comparison for s-series
 n  this_300K  Fang_300K  ratio  this_0K  Fang_1/A  exp  unique
21     12.215        6.8   1.80   19.843      7.6    --    174
22     13.658        8.3   1.65   22.762      9.1   6.5    180
23     15.193        9.8   1.55   25.959     10.7   8.1    186
24     16.821       11.2   1.50   29.446     12.5   9.4    192
25     18.541       13.0   1.43   33.237     14.5  10.6    198
26     20.354       14.9   1.37   37.344     16.7  12.1    204
27     22.260       16.9   1.32   41.781     19.2    --    210
mean ratio to Fang 300 K = 1.51

Comparison for d-series
 n  this_300K  Fang_300K  ratio  this_0K  Fang_1/A  exp  unique
21     15.446        7.8   1.98   28.636      8.6    --    540
22     17.130        9.0   1.90   32.433     10.1   6.4    558
23     18.144       10.6   1.71   32.891     11.7   7.7    576
24     21.270       12.1   1.76   42.863     13.5   8.7    594
25     23.264       13.7   1.70   47.848     15.5   9.8    612
26     25.431       15.9   1.60   5

## Interpretation

This comparison checks whether the repaired `get_nearby` workflow gives the same scale and trend as a known low-`n` Yb lifetime measurement. The current result is useful but not a full validation: it reproduces the monotonic lifetime growth with `n`, but the absolute lifetimes are systematically longer than Fang et al.'s 300 K calculation and experiment.

That means the algorithmic repair fixes the pathological range request, but the current finite final-state model is still missing effective decay strength for this benchmark. The likely causes are incomplete low-lying perturber/final-state treatment, differences between the current MQDT basis and Fang's hydrogenic calculation, and the fact that low-`n` Yb states are exactly where configuration interaction is most important.
